# App-34b : le 15-puzzle, A* et heuristiques admissibles (jumeau C#)

**Navigation** : [<< App-14 ConnectFour C#](App-14-ConnectFour-CSharp.ipynb) | [Index](../README.md)

> **Twin C#** de `App-34-15Puzzle-Admissible-Astar-Python.ipynb` (issue #21752).
> Marathon **.NET / Python** (#4956), volet **Search / Applications**.
> Kernel `.net-csharp` (.NET 9+) -- **aucun NuGet** : tout est en BCL, le carnet ne depend d'aucun reseau.

Le **15-puzzle glissant** est l'exemple par lequel tout cours introduit l'A* : un espace
d'etats discret, un invariant de **resolvabilite** non trivial, et des heuristiques dont la
force se **mesure** (noeuds developpes, cout de solution).

Ce carnet est le jumeau C# de la version Python : meme protocole, meme jeu de mesures, memes
constantes. Il **reconstruit** la semantique depuis zero -- encode, parite, heuristiques,
recherches -- et ne fait aucun appel a la version Python.

In [1]:
using System;
using System.Collections.Generic;
using System.Linq;

static readonly int SEED = 42;
static readonly int N = 4;      // grille 4x4
static readonly int SIZE = N * N;
static readonly int CAP = 3000000;   // plafond d'expansions (garantit l'arret)

Console.WriteLine($"Setup OK. Seed={SEED}, grille {N}x{N} : {SIZE} cases (15 tuiles + 1 case vide).");
Console.WriteLine($"Runtime : {Environment.Version} (.NET)");

The below script needs to be able to find the current output cell; this is an easy method to get it.

Setup OK. Seed=42, grille 4x4 : 16 cases (15 tuiles + 1 case vide).


Runtime : 10.0.13 (.NET)


***
## 1. L'espace d'etats -- (S, s0, A, T, G)

L'etat est une grille 4x4 : les cases 1 a 15 portent une tuile, la case `0` est le vide.
On l'**encode dans un `ulong`** (4 bits par case, `0` = vide) : un etat tient dans un entier,
ce qui rend les tables de hachage triviales et la comparaison d'etats constante.

- **Actions** : les quatre deplacements de la case vide (droite, gauche, bas, haut).
- **Transition** : echange du vide avec la tuile voisine.
- **But** : l'etat `1..15, 0` (rangee par rangee).

L'encodage n'est pas un detail d'implementation : c'est ce qui permet de mesurer des
**centaines de milliers** de noeuds developpes sans que le cout de la structure de donnees
domine la mesure.

In [2]:
// --- Encodage : 4 bits par case, 0 = case vide ---
static ulong Pack(int[] tiles)
{
    ulong s = 0;
    for (int i = 0; i < tiles.Length; i++) s |= (ulong)tiles[i] << (4 * i);
    return s;
}

static int[] Unpack(ulong s)
{
    var t = new int[16];
    for (int i = 0; i < 16; i++) t[i] = (int)((s >> (4 * i)) & 0xF);
    return t;
}

static readonly int[] GOAL = { 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 0 };

static int[] Swap(int[] t, int a, int b)
{
    var c = (int[])t.Clone();
    (c[a], c[b]) = (c[b], c[a]);
    return c;
}

// Quatre deplacements de la case vide : droite, gauche, bas, haut.
// (etat suivant, position du vide avant, position du vide apres)
static IEnumerable<(ulong s, int from, int to)> Neighbours(ulong s)
{
    int[] t = Unpack(s);
    int blank = Array.IndexOf(t, 0);
    int r = blank / 4, c = blank % 4;
    if (c < 3) yield return (Pack(Swap(t, blank, blank + 1)), blank, blank + 1);
    if (c > 0) yield return (Pack(Swap(t, blank, blank - 1)), blank, blank - 1);
    if (r < 3) yield return (Pack(Swap(t, blank, blank + 4)), blank, blank + 4);
    if (r > 0) yield return (Pack(Swap(t, blank, blank - 4)), blank, blank - 4);
}

static bool IsGoal(ulong s) => s == Pack(GOAL);

var s0 = Pack(GOAL);
Console.WriteLine($"Etat but encode sur 8 octets : {s0:X16}");
Console.WriteLine($"Voisins de l'etat but : {Neighbours(s0).Count()} (le vide est dans le coin : 2 deplacements possibles).");

Etat but encode sur 8 octets : 0FEDCBA987654321


Voisins de l'etat but : 2 (le vide est dans le coin : 2 deplacements possibles).


***
## 2. La resolvabilite -- l'invariant de parite

Une configuration du 15-puzzle n'est pas toujours resoluble, et **l'echec est muet** : la
recherche epuise l'espace sans jamais atteindre le but. Le discriminant est la **parite** :

- on compte les **inversions** -- paires de tuiles dont l'ordre est inverse par rapport a l'ordre
  final (la case vide est ignoree) ;
- on ajoute la **ligne de la case vide comptee depuis le bas** (1 a 4) ;
- pour une grille de largeur **paire** (4), la configuration est resoluble ssi la somme est
  **impaire**.

La regle se verifie par la mesure ci-dessous, sur trois proprietes independantes.

In [3]:
// --- Parite des inversions (la case vide est ignoree) ---
static int Inversions(int[] t)
{
    var tiles = t.Where(v => v != 0).ToArray();
    int inv = 0;
    for (int i = 0; i < tiles.Length; i++)
        for (int j = i + 1; j < tiles.Length; j++)
            if (tiles[i] > tiles[j]) inv++;
    return inv;
}

// Grille de largeur paire : resolvable ssi (inversions + ligne depuis le bas) est impair.
static bool Solvable(int[] t)
{
    int rowFromBottom = 4 - (Array.IndexOf(t, 0) / 4);
    return (Inversions(t) + rowFromBottom) % 2 == 1;
}

// Mesure 1 : sur un echantillon aleatoire, la moitie des configurations est resoluble.
var rngParity = new Random(SEED);
int echantillon = 20000, resolubles = 0;
for (int k = 0; k < echantillon; k++)
{
    var perm = Enumerable.Range(0, 16).ToArray();
    for (int i = 15; i > 0; i--)
    {
        int j = rngParity.Next(i + 1);
        (perm[i], perm[j]) = (perm[j], perm[i]);
    }
    if (Solvable(perm)) resolubles++;
}

// Mesure 2 : l'etat but est resolvable.
bool butResolvable = Solvable(GOAL);

// Mesure 3 : un deplacement legal preserve la resolvabilite (il est reversible).
var apresUnCoup = Neighbours(Pack(GOAL)).First();
bool coupLegalPreserve = Solvable(Unpack(apresUnCoup.s));

Console.WriteLine($"Mesure 1 -- echantillon aleatoire de {echantillon} configurations : {resolubles} resolvables ({100.0 * resolubles / echantillon:F1} %)");
Console.WriteLine($"Mesure 2 -- l'etat but est resolvable : {butResolvable}");
Console.WriteLine($"Mesure 3 -- un deplacement legal preserve la resolvabilite : {coupLegalPreserve}");

Mesure 1 -- echantillon aleatoire de 20000 configurations : 10023 resolvables (50,1 %)


Mesure 2 -- l'etat but est resolvable : True


Mesure 3 -- un deplacement legal preserve la resolvabilite : True


***
## 3. Deux heuristiques admissibles, et une recherche non admissible

Une heuristique est **admissible** quand elle ne surestime jamais le cout restant :
`h(s) <= d*(s)`. L'A* est alors **optimal** -- il renvoie le chemin le plus court -- mais son
cout depend de la force de `h`.

- **Manhattan** : somme des distances (verticale + horizontale) de chaque tuile a sa position
  finale. Admissible, mais elle ignore les obstacles *entre* les tuiles.
- **Manhattan + conflit lineaire** : deux tuiles qui occupent la meme ligne que leur position
  finale, mais dans un **ordre inverse**, doivent se croiser. Chaque tuile qui doit **quitter**
  sa ligne (ou sa colonne) puis y revenir coute au moins **2 deplacements de plus**.
  On compte le nombre **minimal** de tuiles a retirer de chaque ligne (ou colonne) pour que
  l'ordre relatif soit correct -- c'est la longueur de la sous-suite croissante la plus
  longue qui reste en place -- et on ajoute `2 x (ce nombre)`. L'heuristique reste admissible,
  mais elle est plus forte.
  **Un piege de double comptage** : une tuile deja a sa place finale ne doit **ni** quitter sa
  ligne **ni** quitter sa colonne. La compter dans les deux groupes lui ferait attribuer
  `+4` deplacements qu'elle ne demande pas -- et l'heuristique cesserait d'etre admissible.
  Le code ci-dessous **exclut** donc les tuiles deja a leur place ; c'est ce qui rend la
  verification d'admissibilite du paragraphe 4 vraie.
- **A* pondere** (`f = g + w * h`, ici `w = 2`) : l'heuristique est **multipliee**, donc
  n'est plus admissible. La recherche devient beaucoup plus rapide, mais le cout de la
  solution peut **degrader**.

Les trois sont mesurees sur les memes puzzles, avec la meme graine.

In [4]:
// --- Heuristique 1 : distance de Manhattan ---
static int Manhattan(int[] t)
{
    int h = 0;
    for (int i = 0; i < 16; i++)
    {
        if (t[i] == 0) continue;
        int goal = t[i] - 1;
        h += Math.Abs(i / 4 - goal / 4) + Math.Abs(i % 4 - goal % 4);
    }
    return h;
}

// --- Longueur de la sous-suite croissante la plus longue (n <= 4) ---
static int LongestIncreasing(int[] seq)
{
    var best = new int[seq.Length];
    int max = 0;
    for (int i = 0; i < seq.Length; i++)
    {
        best[i] = 1;
        for (int j = 0; j < i; j++)
            if (seq[j] < seq[i] && best[j] + 1 > best[i]) best[i] = best[j] + 1;
        if (best[i] > max) max = best[i];
    }
    return max;
}

// --- Heuristique 2 : conflit lineaire (2 points par tuile qui doit sortir) ---
// Pour chaque ligne : les tuiles dont la position finale est sur cette ligne, dans leur
// ordre actuel. Le nombre minimal de tuiles a retirer pour restaurer l'ordre est
// (nombre de tuiles) - (sous-suite croissante la plus longue).
// ATTENTION -- les tuiles deja a leur place finale sont EXCLUES : elles ne doivent ni
// quitter leur ligne ni leur colonne, et les compter dans les deux groupes ferait
// surestimer le cout restant (l'heuristique cesserait d'etre admissible).
static int LinearConflict(int[] t)
{
    int retraits = 0;
    for (int r = 0; r < 4; r++)
    {
        var cols = new List<int>();
        for (int c = 0; c < 4; c++)
        {
            int tile = t[r * 4 + c];
            if (tile == 0) continue;
            int gpos = tile - 1;
            if (gpos / 4 == r && gpos % 4 != c) cols.Add(gpos % 4);
        }
        retraits += cols.Count - LongestIncreasing(cols.ToArray());
    }
    for (int c = 0; c < 4; c++)
    {
        var rows = new List<int>();
        for (int r = 0; r < 4; r++)
        {
            int tile = t[r * 4 + c];
            if (tile == 0) continue;
            int gpos = tile - 1;
            if (gpos % 4 == c && gpos / 4 != r) rows.Add(gpos / 4);
        }
        retraits += rows.Count - LongestIncreasing(rows.ToArray());
    }
    return 2 * retraits;
}

// --- A* : f = g + w * h, file de priorite sur f ---
// Le jeu `closed` compte chaque etat developpe UNE fois : c'est la mesure "noeuds developpes".
static (bool found, int cost, int expanded) Astar(ulong start, Func<int[], int> h, int w = 1, int cap = 3000000)
{
    ulong goal = Pack(GOAL);
    var g = new Dictionary<ulong, int> { [start] = 0 };
    var closed = new HashSet<ulong>();
    var pq = new PriorityQueue<ulong, int>();
    pq.Enqueue(start, w * h(Unpack(start)));
    int expanded = 0;
    while (pq.TryDequeue(out ulong cur, out int prio))
    {
        if (!closed.Add(cur)) continue;
        expanded++;
        if (expanded > cap) return (false, -1, expanded);
        if (cur == goal) return (true, g[cur], expanded);
        foreach (var n in Neighbours(cur))
        {
            ulong s = n.s;
            int ng = g[cur] + 1;
            if (!g.TryGetValue(s, out int old) || ng < old)
            {
                g[s] = ng;
                pq.Enqueue(s, ng + w * h(Unpack(s)));
            }
        }
    }
    return (false, -1, expanded);
}

Console.WriteLine($"Etat but -- Manhattan : {Manhattan(GOAL)} (valeur exacte : 0) -- conflit lineaire : {LinearConflict(GOAL)}");

Etat but -- Manhattan : 0 (valeur exacte : 0) -- conflit lineaire : 0


***
## 4. La mesure -- noeuds developpes x cout de solution

Les puzzles sont produits par un **melange legal** de l'etat but : a chaque coup on deplace la
case vide au hasard, en interdisant de revenir immediatement en arriere. Cette generation est
**garantie solvable** (tout coup legal est reversible) et **deterministe** (graine fixe).

Un melange court laisse le puzzle **près du but** (l'espace de recherche est negligeable) ;
il faut des dizaines de coups pour obtenir des instances ou l'heuristique discrimine
reellement. Le protocole mesure donc quatre profondeurs de melange, trois puzzles chacune, et
compare les trois recherches :

- **A\* Manhattan** -- admissible, sert de **reference optimale** ;
- **A\* Manhattan + conflit lineaire** -- admissible, plus forte ;
- **A\* pondere (w = 2)** -- non admissible : plus rapide, cout potentiellement degrade.

Le cout de reference est celui de l'A* avec Manhattan : l'heuristique etant admissible, c'est
le cout **optimal**. Chaque recherche est plafonnee (3 000 000 d'expansions) : si une recherche
n'aboutit pas, le puzzle est **ecarte entierement** -- les moyennes ne portent que sur les
puzzles que les **trois** recherches ont resolus, car comparer des moyennes calculees sur des
ensembles differents n'aurait aucun sens. Le tableau verifie ensuite, sur ces instances
communes, que le conflit lineaire **n'a pas degrade** le cout optimal (il reste admissible) et
que l'A* pondere, lui, **degrade** le cout sur au moins un puzzle.

In [5]:
// --- Generation : melange legal depuis l'etat but (garanti solvable, deterministe) ---
static ulong Scramble(int moves, Random rng)
{
    ulong s = Pack(GOAL);
    int prevBlank = -1;
    for (int i = 0; i < moves; i++)
    {
        var nbrs = Neighbours(s).ToList();
        var cand = nbrs.Where(n => n.to != prevBlank).ToList();
        if (cand.Count == 0) cand = nbrs;
        var pick = cand[rng.Next(cand.Count)];
        prevBlank = pick.from;
        s = pick.s;
    }
    return s;
}

var rng = new Random(SEED);
int[] profondeurs = { 20, 40, 60, 80 };
int nbPuzzles = 3;

int exclusTotaux = 0;                 // puzzles ecarte (une recherche n'a pas abouti)
bool conflitLineaireOptimale = true;  // le cout A*(Manh+conflit) reste-t-il optimal ?
bool pondereDegrade = false;          // l'A* pondere a-t-il degrade le cout au moins une fois ?
int compares = 0;                     // puzzles compares sur les trois recherches

Console.WriteLine($"profondeur | h moyen | recherche        | noeuds dev. (moy) | cout solution (moy) | ecart au cout optimal");
Console.WriteLine($"-----------+---------+------------------+-------------------+--------------------+--------------------");

foreach (int depth in profondeurs)
{
    var puzzles = new List<ulong>();
    for (int k = 0; k < nbPuzzles; k++) puzzles.Add(Scramble(depth, rng));

    var hPuzzle = new List<int>();
    var rMan = new List<(bool, int, int)>();
    var rConflit = new List<(bool, int, int)>();
    var rPondere = new List<(bool, int, int)>();
    foreach (var p in puzzles)
    {
        hPuzzle.Add(Manhattan(Unpack(p)));
        rMan.Add(Astar(p, Manhattan));
        rConflit.Add(Astar(p, t => Manhattan(t) + LinearConflict(t)));
        rPondere.Add(Astar(p, Manhattan, w: 2));
    }

    // Sous-ensemble COMMUN : seuls les puzzles resolus par les TROIS recherches entrent
    // dans les moyennes. Comparer des moyennes portees sur des ensembles differents
    // n'aurait aucun sens -- et c'est exactement ce qui rendrait l'ecart illisible.
    var communs = new List<int>();
    for (int i = 0; i < nbPuzzles; i++)
        if (rMan[i].Item1 && rConflit[i].Item1 && rPondere[i].Item1) communs.Add(i);

    int exclus = nbPuzzles - communs.Count;
    exclusTotaux += exclus;
    compares += communs.Count;

    long nM = 0, nC = 0, nP = 0;
    int cM = 0, cC = 0, cP = 0, hMoyen = 0;
    foreach (int i in communs)
    {
        nM += rMan[i].Item3; cM += rMan[i].Item2;
        nC += rConflit[i].Item3; cC += rConflit[i].Item2;
        nP += rPondere[i].Item3; cP += rPondere[i].Item2;
        hMoyen += hPuzzle[i];
        if (rConflit[i].Item2 != rMan[i].Item2) conflitLineaireOptimale = false;
        if (rPondere[i].Item2 > rMan[i].Item2) pondereDegrade = true;
    }
    if (communs.Count > 0) hMoyen /= communs.Count;

    string moyNoeuds(long s) => communs.Count == 0 ? "  -- (aucun resolu)" : (s / communs.Count).ToString();
    string moyCout(int s) => communs.Count == 0 ? "  -- (aucun resolu)" : (s / communs.Count).ToString();
    string ecart(int s) => communs.Count == 0 ? "  -- (non comparable)" : (s / communs.Count).ToString();

    Console.WriteLine($"{depth,10} | {hMoyen,7} | A* Manhattan     | {moyNoeuds(nM),17} | {moyCout(cM),18} | {ecart(0),18}");
    Console.WriteLine($"{depth,10} | {hMoyen,7} | A* Manh+conflit  | {moyNoeuds(nC),17} | {moyCout(cC),18} | {ecart(cC - cM),18}");
    Console.WriteLine($"{depth,10} | {hMoyen,7} | A* pondere (w=2) | {moyNoeuds(nP),17} | {moyCout(cP),18} | {ecart(cP - cM),18}");
    if (exclus > 0)
        Console.WriteLine($"{depth,10} | {hMoyen,7} | (puzzles ecarte : une recherche a atteint le plafond de {CAP} expansions)");
}

Console.WriteLine();
Console.WriteLine($"Verification -- puzzles ecarte (une des trois recherches n'a pas abouti) : {exclusTotaux}");
Console.WriteLine($"Verification -- puzzles compares sur les trois recherches : {compares}");
Console.WriteLine($"Verification -- le conflit lineaire reste admissible (cout identique a Manhattan) : {conflitLineaireOptimale}");
Console.WriteLine($"Verification -- l'A* pondere a degrade le cout sur au moins un puzzle : {pondereDegrade}");

profondeur | h moyen | recherche        | noeuds dev. (moy) | cout solution (moy) | ecart au cout optimal


-----------+---------+------------------+-------------------+--------------------+--------------------


        20 |      14 | A* Manhattan     |                80 |                 17 |                  0


        20 |      14 | A* Manh+conflit  |                63 |                 17 |                  0


        20 |      14 | A* pondere (w=2) |                31 |                 17 |                  0


...

        80 |      26 | (puzzles ecarte : une recherche a atteint le plafond de 3000000 expansions)


Verification -- puzzles ecarte (une des trois recherches n'a pas abouti) : 1


Verification -- puzzles compares sur les trois recherches : 11


Verification -- le conflit lineaire reste admissible (cout identique a Manhattan) : True


Verification -- l'A* pondere a degrade le cout sur au moins un puzzle : True


***
## 5. Exercices

Les trois exercices sont des **stubs** : ils rendent une valeur neutre et le carnet s'execute de
bout en bout. A vous de les implementer.

### Exercice 1 -- le test de resolvabilite

Ecrivez le test de parite : compter les inversions (en ignorant la case vide), ajouter la ligne
de la case vide comptee depuis le bas (1 a 4), et conclure.

In [6]:
// Exercice 1 : la configuration est-elle resolvable ?
static bool SolvableEtudiant(int[] t)
{
    // TODO etudiant : compter les inversions de t (case vide ignoree), ajouter la
    // ligne de la case vide comptee depuis le bas, et conclure (impair => resolvable).
    // Indice : la largeur de la grille est paire (4) ; une grille de largeur impaire
    // s'obtient par une autre regle (les inversions seules suffisent).
    return false; // TODO etudiant
}

Console.WriteLine("Exercice 1 : SolvableEtudiant(int[]) definie -- a completer par l'etudiant (renvoie false en attendant).");

Exercice 1 : SolvableEtudiant(int[]) definie -- a completer par l'etudiant (renvoie false en attendant).


### Exercice 2 -- le conflit lineaire au-dessus de Manhattan

Ecrivez le compte des conflits lineaires : pour chaque ligne et chaque colonne, collecter les
tuiles deja placees sur leur ligne (ou colonne) finale, puis compter le nombre **minimal** de
tuiles qu'il faut retirer pour que l'ordre relatif soit correct.

In [7]:
// Exercice 2 : nombre de conflits lineaires (bonde de 2 points) de la configuration.
static int ConflitsLineairesEtudiant(int[] t)
{
    // TODO etudiant : pour chaque ligne (et chaque colonne), collecter les tuiles dont
    // la position finale est sur cette ligne (ou colonne), puis compter le nombre
    // minimal de retraits necessaires pour restaurer un ordre relatif correct.
    // Indice : la tuile 1 a pour position finale (0,0) ; la tuile 15 a pour position
    // finale (2,3) ; le nombre minimal de retraits est (nombre de tuiles) - (longueur
    // de la sous-suite croissante la plus longue) sur les indices de colonne (ou de ligne).
    // Attention : excluez les tuiles deja a leur place finale, sinon elles seraient
    // comptees deux fois et l'heuristique cesserait d'etre admissible.
    return 0; // TODO etudiant
}

Console.WriteLine("Exercice 2 : ConflitsLineairesEtudiant(int[]) definie -- a completer par l'etudiant (renvoie 0 en attendant).");

Exercice 2 : ConflitsLineairesEtudiant(int[]) definie -- a completer par l'etudiant (renvoie 0 en attendant).


### Exercice 3 -- l'ecart d'optimalite d'un A* pondere

Un A* pondere utilise `f = g + w * h`. Pour `w = 1` il est optimal ; pour `w > 1` l'heuristique
n'est plus admissible et la solution trouvee peut couter plus cher. Mesurez l'ecart.

In [8]:
// Exercice 3 : ecart entre le cout d'un A* pondere (poids w) et le cout optimal.
static int EcartOptimaliteEtudiant(int w)
{
    // TODO etudiant : lancer A* avec f = g + w*h sur un puzzle melange (graine fixe),
    // comparer le cout obtenu au cout optimal (A* avec Manhattan, admissible) et
    // retourner la difference.
    // Indice : un poids w > 1 rend h non admissible : le cout peut augmenter, et
    // l'ecart mesure est justement ce que l'on paie en rapidite contre optimalite.
    return 0; // TODO etudiant
}

Console.WriteLine("Exercice 3 : EcartOptimaliteEtudiant(int) definie -- a completer par l'etudiant (renvoie 0 en attendant).");

Exercice 3 : EcartOptimaliteEtudiant(int) definie -- a completer par l'etudiant (renvoie 0 en attendant).


In [9]:
// Les trois stubs s'executent sans lever d'exception : le carnet tourne de bout en bout.
var etatDepart = new int[] { 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15 };
Console.WriteLine($"Exercice 1 (stub) : SolvableEtudiant etat => {SolvableEtudiant(etatDepart)}");
Console.WriteLine($"Exercice 2 (stub) : ConflitsLineairesEtudiant etat => {ConflitsLineairesEtudiant(etatDepart)}");
Console.WriteLine($"Exercice 3 (stub) : EcartOptimaliteEtudiant(2) => {EcartOptimaliteEtudiant(2)}");

Exercice 1 (stub) : SolvableEtudiant etat => False


Exercice 2 (stub) : ConflitsLineairesEtudiant etat => 0


Exercice 3 (stub) : EcartOptimaliteEtudiant(2) => 0


***
## Conclusion

Ce jumeau C# a reconstruit depuis zero le protocole du carnet Python (#21752) :

1. **l'espace d'etats** encode dans un `ulong` (4 bits par case) -- un etat tient dans un entier,
   et la mesure des noeuds developpes ne depend pas de la structure de donnees ;
2. **la resolvabilite** par parite des inversions, verifiee par la mesure sur trois
   proprietes independantes (moitie des configurations aleatoires, etat but, un coup legal) ;
3. **deux heuristiques admissibles** -- Manhattan, et Manhattan + conflit lineaire -- comparees
   sur le **nombre de noeuds developpes** et le **cout de solution** ;
4. **un contraste non admissible** -- l'A* pondere (`w = 2`), qui divise le nombre de noeuds
   developpes mais **degrade le cout** de la solution.

Les trois exercices (parite, conflit lineaire, ecart d'optimalite d'un A* pondere) sont des
stubs : ils rendent une valeur neutre, le carnet s'execute de bout en bout.

**Portee de la parite .NET / Python.** Les deux carnets partagent le **protocole** (meme
encodage 4 bits, meme test de parite, meme jeu de mesures : noeuds developpes et cout de
solution, meme constante de graine `42`). Ils ne partagent **pas** les instances tirees : le
generateur pseudo-aleatoire de .NET et celui de Python ne produisent pas la meme suite, donc
les **nombres** du tableau different d'un runtime a l'autre par construction. Ce qui est
**invariant** entre les deux, et qui est verifie par la mesure dans chaque carnet, ce sont les
**proprietes** : la parite, l'admissibilite du conflit lineaire (cout optimal inchange), et la
degradation du cout par l'A* pondere.